In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('covid_toy.csv')

In [8]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [13]:
from sklearn.model_selection import train_test_split
X_train , X_test, Y_train, Y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'], test_size = 0.2, random_state = 42)

In [16]:
X_test.shape

(20, 5)

## 1. Seapratly for missing value imptation (fever) , ordinal encoding variable(cough) , nominal encoding (gender, city) 

In [18]:
from sklearn.impute import SimpleImputer

In [31]:
df[['fever']]

,fever
0,103.0
1,100.0
2,101.0
3,98.0
4,101.0
...,...
95,104.0
96,101.0
97,101.0
98,98.0


In [26]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])

In [34]:
# ordinal encoding -> cough
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])

In [38]:
# one hot encoding -> gender , city
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop = 'first' , sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

In [46]:
type(X_test_gender_city)

numpy.ndarray

In [57]:
X_train_age = X_train[['age']].values
X_test_age = X_test[['age']].values

In [59]:
import numpy as np

In [61]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

In [64]:
X_test_transformed.shape

(20, 7)

# 2. using column transformer 

In [65]:

from sklearn.compose import ColumnTransformer

In [66]:
transformer = ColumnTransformer(
    transformers = [
        ('trf1', SimpleImputer(), ['fever']), 
        ('trf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('trf3', OneHotEncoder(drop = 'first', sparse_output=False), ['gender', 'city'])
    ]  ,  remainder='passthrough'
)

In [68]:
X_train_direct = transformer.fit_transform(X_train)

In [69]:
X_train_direct.shape

(80, 7)

In [70]:
X_test_direct = transformer.transform(X_test)

In [71]:
X_test_direct.shape

(20, 7)

In [74]:
print(np.array_equal(X_train_transformed, X_train_direct))

False


In [76]:
print(X_train_transformed[0])
print(X_train_direct[0])

[ 81. 101.   0.   0.   0.   1.   0.]
[101.   0.   0.   0.   0.   1.  81.]
